In [19]:
import torch
import torch.nn as nn

class Autoencoder(nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),

            nn.Linear(256, 128),
            nn.ReLU(),

            nn.Linear(128, 2)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(2, 128),
            nn.ReLU(),

            nn.Linear(128, 256),
            nn.ReLU(),

            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):

        z = self.encoder(x)

        reconstructed = self.decoder(z)

        return reconstructed, z

In [20]:
model = Autoencoder()

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [21]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

for epoch in range(20):
    
    if 'train_loader' not in globals():
        transform = transforms.ToTensor()
        train_dataset = datasets.MNIST(root='./data', train=True, transform=transform, download=True)
        globals()['train_loader'] = DataLoader(train_dataset, batch_size=128, shuffle=True)

    train_loader = globals()['train_loader']
    for images, _ in train_loader:

        images = images.view(-1, 784)

        reconstructed, z = model(images)

        loss = criterion(
            reconstructed,
            images
        )

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(
        f"Epoch {epoch+1}, Loss: {loss.item():.4f}"
    )

Epoch 1, Loss: 0.0505
Epoch 2, Loss: 0.0431
Epoch 3, Loss: 0.0423
Epoch 4, Loss: 0.0424
Epoch 5, Loss: 0.0421
Epoch 6, Loss: 0.0398
Epoch 7, Loss: 0.0408
Epoch 8, Loss: 0.0384
Epoch 9, Loss: 0.0427
Epoch 10, Loss: 0.0388
Epoch 11, Loss: 0.0376
Epoch 12, Loss: 0.0361
Epoch 13, Loss: 0.0396
Epoch 14, Loss: 0.0357
Epoch 15, Loss: 0.0364
Epoch 16, Loss: 0.0392
Epoch 17, Loss: 0.0365
Epoch 18, Loss: 0.0349
Epoch 19, Loss: 0.0369
Epoch 20, Loss: 0.0400


In [22]:
latent_vectors = []
labels = []

with torch.no_grad():

    if 'test_loader' not in globals():
        test_dataset = datasets.MNIST(root='./data', train=False, transform=transforms.ToTensor(), download=True)
        globals()['test_loader'] = DataLoader(test_dataset, batch_size=128, shuffle=False)

    test_loader = globals()['test_loader']
    for images, y in test_loader:

        images = images.view(-1, 784)

        _, z = model(images)

        latent_vectors.append(z)
        labels.append(y)

In [23]:
latent_vectors = torch.cat(latent_vectors)
labels = torch.cat(labels)